# Étape 03 — Gestion et registre des modèles

Ce notebook complète les TP 05 et 02. Il sélectionne un modèle déjà suivi dans MLflow, l'ajoute au registre, documente sa version et lui attribue des alias. Il ne réentraîne pas de modèle pour retrouver un artefact.

## 1. Artefact, modèle enregistré, version, alias et déploiement

| Élément | Rôle | Exemple Yellow Taxi |
| --- | --- | --- |
| Artefact de modèle | Fichier produit par une exécution, avec ses paramètres et métriques | Modèle XGBoost d'un run donné |
| Modèle enregistré | Nom stable qui regroupe des versions | `nyc-taxi-duration-model` |
| Version | Référence à un artefact et à son run source | Version 1 ou 2 du modèle enregistré |
| Alias | Nom indiquant le rôle actuel d'une version | `champion` ou `challenger` |
| Déploiement | Mise à disposition du modèle dans une application | Une API charge `champion` |

Le registre organise les versions et leurs métadonnées ; il ne déploie pas le modèle. Déplacer un alias ne supprime pas les versions précédentes.

## 2. Niveau de maturité

Cette étape reste au niveau 2, entraînement automatisé. Le registre améliore la traçabilité et prépare le niveau 3, mais ne constitue pas à lui seul un processus de déploiement automatisé.

## 3. Prérequis — serveur SQLite du TP précédent

Dans un terminal à la racine du dépôt, activez l'environnement `mlops`, puis démarrez le serveur. Laissez ce terminal ouvert : le registre a besoin d'un backend de base de données et le stockage local par fichiers du scénario 1 ne suffit pas.

```bash
conda activate mlops
mlflow server \
  --host 0.0.0.0 \
  --port 5000 \
  --backend-store-uri sqlite:///mlflow.db \
  --serve-artifacts \
  --artifacts-destination ./mlartifacts
```

Si le serveur tourne déjà pour le TP 02, gardez-le actif et continuez ci-dessous. Dans Codespaces, l'interface MLflow est accessible via le port 5000, en visibilité **Private**.

In [1]:
from pathlib import Path

import mlflow
import pandas as pd
from mlflow.tracking import MlflowClient
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import root_mean_squared_error
from sklearn.pipeline import Pipeline

tracking_uri = "http://127.0.0.1:5000"
experiment_name = "nyc-taxi-duration"
registered_model_name = "nyc-taxi-duration-model"

mlflow.set_tracking_uri(tracking_uri)
mlflow.set_registry_uri(tracking_uri)
client = MlflowClient(tracking_uri=tracking_uri, registry_uri=tracking_uri)

print("Tracking URI :", mlflow.get_tracking_uri())
print("Registry URI :", mlflow.get_registry_uri())

Tracking URI : http://127.0.0.1:5000
Registry URI : http://127.0.0.1:5000


## 4. Vérifier les expériences

Le client interroge le serveur SQLite, plutôt que le stockage local par fichiers.

In [2]:
experiments = client.search_experiments()

for experiment_item in experiments:
    print(experiment_item.experiment_id, experiment_item.name)

1 nyc-taxi-duration
0 Default


## 5. Retrouver la meilleure exécution qui contient un modèle

Le classement utilise la RMSE de validation la plus faible, comme dans le TP 02. MLflow 3.18 référence les modèles enregistrés comme objets `LoggedModel` (URI `models:/m-...`) plutôt que comme un dossier `model` directement listable parmi les artefacts du run. Le `LoggedModel` conserve le lien vers son run source ; on vérifie son état avant de l'enregistrer.

In [3]:
experiment = mlflow.get_experiment_by_name(experiment_name)
if experiment is None:
    raise ValueError(f"Expérience introuvable : {experiment_name}")

runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string="attributes.status = 'FINISHED'",
    order_by=["metrics.validation_rmse ASC"],
    max_results=100,
)
if runs.empty:
    raise ValueError(
        "Aucune exécution terminée. Enregistrez d'abord un run avec le serveur SQLite du TP 02."
    )

logged_models = client.search_logged_models(
    experiment_ids=[experiment.experiment_id],
    max_results=100,
)
logged_models_by_run = {
    logged_model.source_run_id: logged_model
    for logged_model in logged_models
    if logged_model.name == "model" and logged_model.status.value == "READY"
}
runs_with_model = runs[runs["run_id"].isin(logged_models_by_run)].copy()

if runs_with_model.empty:
    raise ValueError(
        "Aucun run terminé ne contient de modèle MLflow prêt nommé 'model'. "
        "Vérifiez les runs du TP 02."
    )

runs_with_model = runs_with_model.sort_values("metrics.validation_rmse", ascending=True)
display(runs_with_model[["run_id", "params.model", "metrics.validation_rmse"]])

best_run = runs_with_model.iloc[0]
run_id = best_run["run_id"]
best_logged_model = logged_models_by_run[run_id]
run_model_uri = f"runs:/{run_id}/model"
model_uri = best_logged_model.model_uri
print("Run retenu :", run_id)
print("URI de l'artefact du run :", run_model_uri)
print("URI LoggedModel MLflow 3 :", model_uri)

,run_id,params.model,metrics.validation_rmse
0,4512bedefeb94b93a1c17ef9416ed94f,XGBoost,4.35197
1,b63d9305897d4be9a91e02df4a0e3a87,XGBoost,4.35197
2,5d12bb9a8aa3475fbae765dd76585e54,XGBoost,4.35197


Run retenu : 4512bedefeb94b93a1c17ef9416ed94f
URI de l'artefact du run : runs:/4512bedefeb94b93a1c17ef9416ed94f/model
URI LoggedModel MLflow 3 : models:/m-2d8994408eea45e8bab87e286d6e90e4


## 6. Enregistrer une version du modèle

L'enregistrement crée le modèle s'il n'existe pas, puis ajoute une version liée au run source. Si vous relancez le notebook après un redémarrage du noyau, une version déjà liée au même run est réutilisée.

In [4]:
existing_versions = client.search_model_versions(
    filter_string=f"name = '{registered_model_name}'"
)
model_version = next(
    (item for item in existing_versions if item.run_id == run_id),
    None,
)
if model_version is None:
    model_version = mlflow.register_model(
        model_uri=model_uri,
        name=registered_model_name,
    )

version = model_version.version
print("Nom :", model_version.name)
print("Version :", version)
print("Run source :", model_version.run_id)

Nom : nyc-taxi-duration-model
Version : 2
Run source : 4512bedefeb94b93a1c17ef9416ed94f


Lister les versions permet de vérifier que chacune conserve son run source.

In [5]:
versions = client.search_model_versions(
    filter_string=f"name = '{registered_model_name}'"
)

for item in sorted(versions, key=lambda version_item: int(version_item.version)):
    print(
        "version:", item.version,
        "run:", item.run_id,
        "status:", item.status,
    )

version: 1 run: 4512bedefeb94b93a1c17ef9416ed94f status: READY
version: 2 run: 4512bedefeb94b93a1c17ef9416ed94f status: READY


## 7. Documenter la version

Ajoutez une description utile et un tag structuré.

In [6]:
client.update_model_version(
    name=registered_model_name,
    version=version,
    description=(
        "Modèle candidat pour prédire la durée des trajets Yellow Taxi. "
        f"Run source {run_id}, RMSE de validation "
        f"{best_run['metrics.validation_rmse']:.4f} minutes."
    ),
)

client.set_model_version_tag(
    name=registered_model_name,
    version=version,
    key="validation_status",
    value="reviewed",
)

version_details = client.get_model_version(registered_model_name, version)
print(version_details.description)
print(version_details.tags)

Modèle candidat pour prédire la durée des trajets Yellow Taxi. Run source 4512bedefeb94b93a1c17ef9416ed94f, RMSE de validation 4.3520 minutes.
{'validation_status': 'reviewed'}


## 8. Attribuer et retrouver l'alias `champion`

Un alias indique le rôle d'une version ; il ne démarre pas de déploiement.

In [7]:
client.set_registered_model_alias(
    name=registered_model_name,
    alias="champion",
    version=version,
)

champion_version = client.get_model_version_by_alias(
    name=registered_model_name,
    alias="champion",
)
print(champion_version.version, champion_version.run_id)

2 4512bedefeb94b93a1c17ef9416ed94f


## 9. Préparer les mêmes données de validation que dans le TP 05

Le modèle du TP 05 a été ajusté sur une matrice issue de `DictVectorizer`. Le notebook de suivi peut aussi enregistrer un `Pipeline` contenant ce vectoriseur. Le code ci-dessous prépare les mêmes dictionnaires et la même matrice : on pourra charger correctement l'un ou l'autre format sans réentraîner de modèle.

In [8]:
import gc

current_dir = Path.cwd()
repo_root = current_dir.parent if current_dir.name == "notebooks" else current_dir
data_path = repo_root / "data" / "yellow_tripdata_2026-05.parquet"
if not data_path.exists():
    raise FileNotFoundError(f"Fichier Parquet introuvable : {data_path}")

columns = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "PULocationID",
    "DOLocationID",
    "trip_distance",
]
df = pd.read_parquet(data_path, columns=columns)
df["duration"] = (
    df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]
).dt.total_seconds() / 60
df = df.loc[df["duration"].between(1, 60)].copy()

categorical = ["PULocationID", "DOLocationID"]
df[categorical] = df[categorical].astype(str)
df["PU_DO"] = df["PULocationID"] + "_" + df["DOLocationID"]

sample_size = min(200_000, len(df))
df = df.sample(n=sample_size, random_state=42)
df = df.sort_values("tpep_pickup_datetime").reset_index(drop=True)
split_index = int(len(df) * 0.8)
df_train = df.iloc[:split_index]
df_val = df.iloc[split_index:]

features = ["PU_DO", "trip_distance"]
training_rows = (
    {"PU_DO": pu_do, "trip_distance": distance}
    for pu_do, distance in zip(df_train["PU_DO"], df_train["trip_distance"])
)
dv = DictVectorizer()
dv.fit(training_rows)
val_dicts = df_val[features].to_dict(orient="records")
X_val = dv.transform(val_dicts).tocsc()
y_val = df_val["duration"].to_numpy(copy=True)

X_val.indices = X_val.indices.astype("int32")
X_val.indptr = X_val.indptr.astype("int32")

validation_size = len(y_val)
del training_rows, df_train, df_val, df
gc.collect()
print("Taille validation :", validation_size)

Taille validation : 40000


## 10. Charger `champion` avec son alias

Le modèle est récupéré par URI d'alias. Pour rester compatible avec les artefacts du TP 02, la cellule détecte si le run contient un `Pipeline` (qui prend des dictionnaires) ou le modèle seul (qui prend la matrice `X_val`).

In [9]:
champion_uri = f"models:/{registered_model_name}@champion"
champion_model = mlflow.sklearn.load_model(champion_uri)

def predict_validation(model):
    model_input = val_dicts if isinstance(model, Pipeline) else X_val
    return model.predict(model_input)

champion_predictions = predict_validation(champion_model)
print("URI :", champion_uri)
print("Prédictions (5 premières) :", champion_predictions[:5])

URI : models:/nyc-taxi-duration-model@champion
Prédictions (5 premières) : [14.270998 12.417974 13.149687  5.954787  5.954787]


## 11. Parcourir le registre et télécharger un artefact

Le téléchargement copie l'artefact du run dans un dossier temporaire local ; le chargement via `models:/...@champion` reste préférable pour l'utilisation courante.

In [10]:
registered_models = client.search_registered_models()
for registered_model in registered_models:
    print(registered_model.name)

download_path = mlflow.artifacts.download_artifacts(
    artifact_uri=best_logged_model.model_uri,
)
print("Artefact téléchargé dans :", download_path)

nyc-taxi-duration-model
Artefact téléchargé dans : /tmp/tmp4f4n_loy


## 12. Exercice — enregistrer et comparer une version `challenger`

Le code choisit un autre run terminé contenant un artefact `model`, l'enregistre sous le même nom, lui ajoute une description et un tag, puis lui attribue l'alias `challenger`. Il compare les deux versions sur le même jeu de validation. La règle de promotion est une RMSE strictement plus faible ; sinon, `champion` reste inchangé.

In [11]:
other_runs = runs_with_model[runs_with_model["run_id"] != run_id]
if other_runs.empty:
    raise ValueError(
        "Il faut deux runs terminés avec l'artefact 'model' pour comparer "
        "champion et challenger. Enregistrez un autre run via le TP 02."
    )

challenger_run = other_runs.iloc[0]
challenger_run_id = challenger_run["run_id"]
challenger_logged_model = logged_models_by_run[challenger_run_id]
challenger_model_uri = challenger_logged_model.model_uri
existing_versions = client.search_model_versions(
    filter_string=f"name = '{registered_model_name}'"
)
challenger_version = next(
    (item for item in existing_versions if item.run_id == challenger_run_id),
    None,
)
if challenger_version is None:
    challenger_version = mlflow.register_model(
        model_uri=challenger_model_uri,
        name=registered_model_name,
    )
challenger_version_number = challenger_version.version

client.update_model_version(
    name=registered_model_name,
    version=challenger_version_number,
    description=(
        "Version challenger pour comparaison sur le jeu de validation Yellow Taxi. "
        f"Run source {challenger_run_id}, RMSE "
        f"{challenger_run['metrics.validation_rmse']:.4f} minutes."
    ),
)
client.set_model_version_tag(
    name=registered_model_name,
    version=challenger_version_number,
    key="validation_status",
    value="reviewed",
)
client.set_registered_model_alias(
    name=registered_model_name,
    alias="challenger",
    version=challenger_version_number,
)

print("Run challenger :", challenger_run_id)
print("Version challenger :", challenger_version_number)

Registered model 'nyc-taxi-duration-model' already exists. Creating a new version of this model...
2026/11/27 14:04:44 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: nyc-taxi-duration-model, version 3
Created version '3' of model 'nyc-taxi-duration-model'.


Run challenger : b63d9305897d4be9a91e02df4a0e3a87
Version challenger : 3


In [12]:
champion_model = mlflow.sklearn.load_model(
    f"models:/{registered_model_name}@champion"
)
challenger_model = mlflow.sklearn.load_model(
    f"models:/{registered_model_name}@challenger"
)

champion_rmse = root_mean_squared_error(y_val, predict_validation(champion_model))
challenger_rmse = root_mean_squared_error(y_val, predict_validation(challenger_model))
print(f"Champion RMSE : {champion_rmse:.4f}")
print(f"Challenger RMSE : {challenger_rmse:.4f}")

if challenger_rmse < champion_rmse:
    client.set_registered_model_alias(
        name=registered_model_name,
        alias="champion",
        version=challenger_version_number,
    )
    print("Promotion : le challenger devient champion.")
else:
    print("Promotion refusée : l'alias champion reste sur sa version actuelle.")

current_champion = client.get_model_version_by_alias(
    name=registered_model_name,
    alias="champion",
)
reloaded_champion = mlflow.sklearn.load_model(
    f"models:/{registered_model_name}@champion"
)
print("Version champion actuelle :", current_champion.version)
print("Run associé :", current_champion.run_id)
print("Prédictions rechargées :", predict_validation(reloaded_champion)[:5])

Champion RMSE : 4.3520
Challenger RMSE : 4.3520
Promotion refusée : l'alias champion reste sur sa version actuelle.
Version champion actuelle : 2
Run associé : 4512bedefeb94b93a1c17ef9416ed94f
Prédictions rechargées : [14.270998 12.417974 13.149687  5.954787  5.954787]


## 13. Vérifier dans l'interface et retenir

Dans l'interface MLflow, ouvrez **Models**, puis `nyc-taxi-duration-model`. Vérifiez les liens vers les runs source, la métrique, la description, le tag `validation_status` et les alias `champion` / `challenger`.

Une exécution garde la trace d'un entraînement ; une version du registre donne un nom stable à son modèle ; un alias désigne la version utilisée par un consommateur. L'alias peut être déplacé ou restauré sans supprimer les anciennes versions. Cela prépare un déploiement, mais ne le réalise pas.